# Stage 12: Cross-Species Transfer and Exercise-Axis Preservation

> **Outputs read: the active config** (printed by the first code cell). The default since 2026-11-15 is **refs5**, the
> registered rerun of Stages 8–14 after the 2026-11-11 audit (`docs/audit/remediation/STEP3_RESULTS.md`): the 15
> references rebuilt into `references_v5/` (HYPOTH re-annotated, the HEART merge applied), the Stage 8 exclusion list
> rebuilt in Ensembl 115, the target sum set by the registered formula, and the repaired gate, DE and network code.
> **Model: retrain v2** (phase 2 final). The default before refs5 (`_refs4all_rebuild` folders) is
> `config/profiles/refs4all_rebuild.yaml`; this notebook as executed on it is in git history at commit `12f94b9`
> (the 14-tissue refs4 version at `542b454`). Earlier comparisons: `deconvolution/ALL_TISSUE_RERUN_RESULTS.md`
> (refs4 against all tissues), `deconvolution/REFS4_RERUN_RESULTS.md` (July against refs4) and
> `docs/RETRAIN_V2_RESULTS.md` (April against v2).
> The transfer comparison uses the gate's method on the human side, over the rat gate's family of blocks
> (`<genecompass_input_dir>_human/gate_rcv10/`).

## Stage Objective

Project rat pseudo-cell expression into the human GeneCompass token space and test whether rat exercise-associated latent axes remain detectable.

## Workflow

| Step | Operation | Analytical purpose |
|------|-----------|--------------------|
| 1 | Ortholog projection | Map rat expression features to eligible human ortholog tokens and quantify mapped expression mass. |
| 2 | Human-space embedding | Embed projected pseudo-cells with the same model using the human species identifier. |
| 3 | Transfer validity | Evaluate preservation of the sex axis as a pre-specified positive-control gate. |
| 4 | Exercise-axis comparison | Compare rat- and human-space supervised-probe metrics for all blocks and Stage 11 hotspots. |

## Inputs and Outputs

| Component | Description |
|-----------|-------------|
| Inputs | Stage 8 pseudo-cells, Stage 3 ortholog maps, Stage 7 model, and Stage 11 hotspot definitions. |
| Primary outputs | Human-space embeddings, transfer-comparison statistics, and hotspot preservation classifications. |

## Result Overview

The question is whether the rat-space exercise hotspots keep their exercise signal in human space, with the sex axis as the positive control; the hotspot classifications and the sex check are computed below. Whether transferred cells land on their real-human counterparts was tested separately against Tabula Sapiens on the default before refs5 (`translation/ATLAS_PROJECTION_16_RESULTS.md`, 16 tissues; `translation/ATLAS_PROJECTION_RESULTS.md`, 14 tissues). That projection has not been repeated on the refs5 cells.

The sections below present the computational evidence in execution order. Every figure is printed by a code cell from the recorded stage outputs or the config; the markdown describes methods and points to those cells rather than quoting numbers.

In [1]:
import pathlib, os
from nb_config import GC_HUMAN_DIR as GCIH
GATE = GCIH / "gate_rcv10"   # transfer comparison by the gate method
from nb_config import tissues
print("=== human-space embedding files ===")
total_cells = 0
for t in tissues:
    emb = GCIH / t / "embeddings" / "cell_embeddings.npy"
    if emb.exists():
        import numpy as np
        arr = np.load(str(emb), mmap_mode="r")
        total_cells += arr.shape[0]
        print(f"  {t:<8}: {arr.shape[0]:5d} cells × {arr.shape[1]} dims  "
              f"({emb.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  {t:<8}: MISSING")

print(f"  TOTAL: {total_cells} human-space embeddings")

print("\n=== comparison outputs ===")
for fn in ["subspace_probe.tsv","transfer_comparison.tsv","transfer_comparison.md"]:
    p = GATE / fn
    sz = f"{p.stat().st_size/1024:.1f} KB" if p.exists() else "MISSING"
    print(f"  {fn}: {sz}")


config: (default config)
Stage 8: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/results_refs5/motrpac
Stage 9: /depot/reese18/apps/motrpac-genecompass/data/deconvolution/genecompass_input_v2_refs5
16 tissues: BAT, BLOOD, CORTEX, HEART, HIPPOC, HYPOTH, KIDNEY, LIVER, LUNG, OVARY, SKMGN, SKMVL, SMLINT, SPLEEN, TESTES, WATSC
=== human-space embedding files ===
  bat     :   300 cells × 768 dims  (0.9 MB)
  blood   :   650 cells × 768 dims  (2.0 MB)
  cortex  :   600 cells × 768 dims  (1.8 MB)
  heart   :   750 cells × 768 dims  (2.3 MB)
  hippoc  :   550 cells × 768 dims  (1.7 MB)
  hypoth  :   550 cells × 768 dims  (1.7 MB)
  kidney  :   750 cells × 768 dims  (2.3 MB)
  liver   :   400 cells × 768 dims  (1.2 MB)
  lung    :  1009 cells × 768 dims  (3.1 MB)
  ovary   :   432 cells × 768 dims  (1.3 MB)
  skmgn   :   450 cells × 768 dims  (1.4 MB)
  skmvl   :   450 cells × 768 dims  (1.4 MB)
  smlint  :   650 cells × 768 dims  (2.0 MB)
  spleen  :   700 cells × 768 dims  (2.2 M

## Steps 1-2: Ortholog Projection and Human-Space Embedding

Each rat gene is re-expressed through its Stage 3 rat-to-human ortholog; eligible human features must also occur in
the human token vocabulary and median dictionary. The next cell prints, per tissue, the rat genes, the share with an
ortholog, the eligible human features, the share of BayesPrism posterior expression (count mass) retained after
projection, and the pseudo-cells embedded (those under the minimum gene count are dropped). Human-space embeddings
use the same v2 model as Stage 9 with species identifier 0.

How far does re-expression as human move each cell? The cell after it computes, per tissue, the mean and median cosine between
a pseudo-cell's rat-space and human-space embedding (paired by `cell_id`, as `translation/transfer_cosine.py` does),
and the median cosine between two different pseudo-cells of the same tissue in rat space. When the second is higher,
a cell's human-space embedding is farther from its rat embedding than other cells of its tissue are: the cosine mostly
measures how far apart the species' regions of the embedding sit, not whether the cell keeps its identity. Whether
the exercise signal survives is the comparison below; identity was tested against Tabula Sapiens on
the default before refs5 (`translation/ATLAS_PROJECTION_16_RESULTS.md`).

In [2]:
import numpy as np, matplotlib.pyplot as plt, pathlib

# Per-tissue transfer stats, read from Stage 12's outputs: rat_genes, pct_ortho and count_mass from
# <human root>/<tissue>/transfer_summary.json; eligible from tokenize_summary.json; n_cells = rows of cell_embeddings.npy
from nb_config import tissues as _tissues, transfer_summary, tokenize_summary, n_embedded
TRANSFER = {}
for t in _tissues:
    tr, tk = transfer_summary(t), tokenize_summary(t, human=True)
    TRANSFER[t] = {"rat_genes": tr["n_rat_genes"], "pct_ortho": round(100 * tr["frac_rat_mapped"], 1),
                   "eligible": tk["n_eligible_genes"], "count_mass": round(100 * tr["frac_count_mass_mapped"], 1),
                   "n_cells": n_embedded(t, human=True)}
PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "ovary":"#EC407A","spleen":"#B7950B",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}
tissues = list(TRANSFER.keys())
x = np.arange(len(tissues))
w = 0.38

pct_ortho  = [TRANSFER[t]["pct_ortho"]   for t in tissues]
count_mass = [TRANSFER[t]["count_mass"]   for t in tissues]
n_cells    = [TRANSFER[t]["n_cells"]      for t in tissues]
colors     = [PALETTE[t] for t in tissues]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), constrained_layout=True)

# Left: dual bar — pct_ortho + count_mass
ax = axes[0]
ax.bar(x - w/2, pct_ortho,  w, color=colors, alpha=0.9, label="Gene mapping (%)", edgecolor="k", lw=0.4)
ax.bar(x + w/2, count_mass, w, color=colors, alpha=0.55,label="Count-mass (%)",  edgecolor="k", lw=0.4, hatch="///")
ax.axhline(80, color="gray", lw=1.0, ls="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels([t.upper() for t in tissues], rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Percentage (%)", fontsize=10)
ax.set_title("Ortholog gene mapping per tissue\n(solid=gene%, hatched=count-mass%)", fontsize=11)
ax.set_ylim(70, 92)
ax.legend(fontsize=9)

for i, (pg, cm) in enumerate(zip(pct_ortho, count_mass)):
    ax.text(i - w/2, pg + 0.15, f"{pg:.1f}", ha="center", fontsize=7, rotation=0)
    ax.text(i + w/2, cm + 0.15, f"{cm:.1f}", ha="center", fontsize=7, rotation=0)

# Right: total pseudo-cells per tissue (horizontal bar, sorted)
ax = axes[1]
order = np.argsort(n_cells)[::-1]
t_ord = [tissues[i] for i in order]
n_ord = [n_cells[i] for i in order]
c_ord = [PALETTE[t] for t in t_ord]
ax.barh(range(len(tissues)), n_ord, color=c_ord, alpha=0.8, edgecolor="k", lw=0.4)
for j,(t,n) in enumerate(zip(t_ord, n_ord)):
    ax.text(n + 15, j, f"{n}", va="center", fontsize=8.5)
ax.set_yticks(range(len(tissues)))
ax.set_yticklabels([t.upper() for t in t_ord], fontsize=9)
ax.set_xlabel("Human-space pseudo-cells", fontsize=10)
ax.set_title(f"Pseudo-cells transferred to human space\n(total {sum(n_cells):,} across {len(tissues)} tissues)", fontsize=11)
ax.set_xlim(0, max(n_cells) * 1.14)

plt.show()
print(f"total human-space pseudo-cells: {sum(n_cells)}")
print(f"mean gene mapping: {np.mean(pct_ortho):.1f}%  count-mass: {np.mean(count_mass):.1f}%")
import pickle, pandas as pd
from nb_config import PROJECT
_r2h = pickle.load(open(PROJECT / "data/training/ortholog_mappings/rat_to_human_mapping.pickle", "rb"))
print(f"Stage 3 rat-to-human map: {len(_r2h):,} rat genes")
_tt = pd.DataFrame(TRANSFER).T
_tt["pseudo-cells"] = [tokenize_summary(t, human=True).get("n_pseudocells", 0) + tokenize_summary(t, human=True).get("n_dropped_lt_min_genes", 0) for t in _tt.index]
print(_tt.rename(columns={"rat_genes": "rat genes", "pct_ortho": "orthologs %", "count_mass": "count mass %", "n_cells": "embedded"}).to_string())

total human-space pseudo-cells: 9041
mean gene mapping: 82.7%  count-mass: 83.9%


Stage 3 rat-to-human map: 15,234 rat genes
        rat genes  orthologs %  eligible  count mass %  embedded  pseudo-cells
bat       14102.0         82.8   11556.0          83.9     300.0           300
blood     14811.0         81.8   11942.0          79.8     650.0           650
cortex    15309.0         82.2   12448.0          86.6     600.0           600
heart     15765.0         81.7   12717.0          84.9     750.0           750
hippoc    15243.0         81.8   12306.0          86.7     550.0           550
hypoth    13985.0         83.3   11566.0          85.9     550.0           550
kidney    14833.0         82.1   12024.0          84.5     750.0           750
liver     12701.0         83.3   10464.0          81.6     400.0           400
lung      15158.0         82.7   12405.0          84.8    1009.0          1050
ovary     14096.0         83.0   11572.0          84.0     432.0           432
skmgn     13093.0         83.5   10835.0          86.4     450.0           450
skmvl    

In [3]:
import numpy as np
from datasets import load_from_disk
from sklearn.preprocessing import normalize
from nb_config import GC_DIR, GC_HUMAN_DIR
rows = []
for t in _tissues:
    rid = load_from_disk(str(GC_DIR / t / "dataset"))["cell_id"]
    hid = load_from_disk(str(GC_HUMAN_DIR / t / "dataset"))["cell_id"]
    er = normalize(np.load(GC_DIR / t / "embeddings" / "cell_embeddings.npy"))
    eh = normalize(np.load(GC_HUMAN_DIR / t / "embeddings" / "cell_embeddings.npy"))
    pos = {k: i for i, k in enumerate(rid)}
    paired = (er[[pos[k] for k in hid]] * eh).sum(1)
    S = er @ er.T
    rows.append({"tissue": t, "paired": len(hid), "rat-human mean": paired.mean(), "rat-human median": np.median(paired),
                 "rat-rat, different cells, median": np.median(S[np.triu_indices(len(er), 1)])})
cos = pd.DataFrame(rows).set_index("tissue")
print(cos.round(3).to_string())
print(f"\nmedian over tissues: rat-human mean {cos['rat-human mean'].median():.3f}, median {cos['rat-human median'].median():.3f}; "
      f"rat-rat between different cells {cos['rat-rat, different cells, median'].median():.3f}")

/depot/reese18/apps/motrpac-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


        paired  rat-human mean  rat-human median  rat-rat, different cells, median
tissue                                                                            
bat        300           0.960             0.959                             0.998
blood      650           0.985             0.985                             0.991
cortex     600           0.962             0.976                             0.989
heart      750           0.961             0.963                             0.996
hippoc     550           0.963             0.971                             0.992
hypoth     550           0.964             0.967                             0.993
kidney     750           0.974             0.974                             0.995
liver      400           0.978             0.980                             0.997
lung      1009           0.983             0.984                             0.991
ovary      432           0.979             0.982                             0.992
skmg

## Steps 3-4: Transfer Validity and Exercise-Axis Comparison

The gate's probe is run on the human-space embedding of the same cells by the same method as the rat side
(`compare_transfer.py`). The human test reuses the rat test's cells, labels, folds and seeds: it shows whether the
model's human-species reading of the rat data keeps the structure, and it is not a replication. Each block gets:

- `status`, the registered rule: PRESERVED (human-space trained AUC ≥ 0.65 with a raw permutation p < 0.05),
  WEAKENED (AUC ≥ 0.65 with p ≥ 0.05) or LOST (AUC < 0.65). Its p is not corrected for the number of blocks.
- `transfer_corrected`, the corrected rule (`docs/audit/remediation/STEP3_PREREG.md` 7): PASS if the block is a
  hotspot or borderline in human space, by Benjamini-Hochberg over the rat gate's analysable blocks with the classes
  of Stage 11 (`q_sup_trained_human`, `gate_class_human`), and its human AUC is at least 0.65.
- `dose_trend_lost`: neither rule uses the dose axis, so each row carries the rat- and human-space dose correlations,
  and the flag is set when the human value is zero or negative.

Sex is the positive control: the human-space median sex AUC must reach the floor of 0.70, and block by block the sex
AUC should carry over (single-sex blocks have no sex AUC). The floor is set in the script; no registration states
it. The next cell prints `compare_transfer.py`'s verdict and the medians; the refs4 and July-reference values are in `deconvolution/ALL_TISSUE_RERUN_RESULTS.md` and
`deconvolution/REFS4_RERUN_RESULTS.md`. Exercise-axis preservation is presented as descriptive evidence.

In [4]:
from nb_config import GATE_HUMAN_DIR
for line in open(GATE_HUMAN_DIR / "transfer_comparison.md"):
    if line.startswith("- ") and ("sup_sex_auc" in line or "transfer-validity" in line or "Spearman(rat, human)" in line):
        print(line.rstrip())

- median sup_sex_auc: rat 0.756 -> human 0.745  (Spearman rat~human across 170 blocks: r=0.932)
- **transfer-validity: PASS** -- a strong, transfer-agnostic biological axis remains detectable in human space (floor 0.70). Embedding biology preserved; exercise claims below are interpretable.
- trained AUC  Spearman(rat, human) = 0.732  (n=193)
- ordinal dose rho Spearman(rat, human) = 0.794  (n=193)


In [5]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import spearmanr

from nb_config import GATE_HUMAN_DIR
GCIH = str(GATE_HUMAN_DIR)
tc   = pd.read_csv(f"{GCIH}/transfer_comparison.tsv", sep="\t")

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "ovary":"#EC407A","spleen":"#B7950B",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)

# Left: sex axis — rat vs human sup_sex_auc
ax = axes[0]
for t in sorted(tc["tissue"].unique()):
    sub = tc[tc["tissue"]==t]
    ax.scatter(sub["sup_sex_auc_rat"], sub["sup_sex_auc_human"],
               c=PALETTE.get(t,"gray"), s=22, alpha=0.6, label=t.upper())

rho, pval = spearmanr(tc["sup_sex_auc_rat"], tc["sup_sex_auc_human"], nan_policy="omit")  # TESTES, OVARY: single-sex, no sex AUC
lo = min(tc["sup_sex_auc_rat"].min(), tc["sup_sex_auc_human"].min()) - 0.03
hi = max(tc["sup_sex_auc_rat"].max(), tc["sup_sex_auc_human"].max()) + 0.03
ax.plot([lo,hi],[lo,hi], "k--", lw=1.0, alpha=0.4, label="Identity")
ax.axhline(0.70, color="#27AE60", lw=1.3, ls=":", alpha=0.8, label="Validity floor (0.70)")
ax.set_xlabel("Rat sex AUC (PLS-1, species=2)", fontsize=10)
ax.set_ylabel("Human sex AUC (PLS-1, species=0)", fontsize=10)
ax.set_title(f"Sex-axis positive control\nSpearman r={rho:.3f} (p={pval:.2e}, n={tc['sup_sex_auc_human'].notna().sum()})", fontsize=11)
med_r = tc["sup_sex_auc_rat"].median()
med_h = tc["sup_sex_auc_human"].median()
ax.text(0.98, 0.04, f"rat median: {med_r:.3f}\nhuman median: {med_h:.3f}\nscript check (human median >= 0.70): {'PASS' if med_h >= 0.70 else 'FAIL'}",
        transform=ax.transAxes, ha="right", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FDEDEC", edgecolor="#C0392B"))
ax.legend(fontsize=7, ncol=2, loc="upper left")

# Right: global fidelity — rat trained AUC vs human trained AUC
ax = axes[1]
hot = tc["is_hotspot"] == True
for t in sorted(tc["tissue"].unique()):
    sub = tc[tc["tissue"]==t]
    m   = sub["is_hotspot"] == True
    ax.scatter(sub.loc[~m,"sup_trained_auc_rat"], sub.loc[~m,"sup_trained_auc_human"],
               c=PALETTE.get(t,"gray"), s=22, alpha=0.5, label=t.upper())
    ax.scatter(sub.loc[m,"sup_trained_auc_rat"],  sub.loc[m,"sup_trained_auc_human"],
               c=PALETTE.get(t,"gray"), s=75, edgecolors="k", lw=1.2, zorder=5)

rho2, pval2 = spearmanr(tc["sup_trained_auc_rat"], tc["sup_trained_auc_human"], nan_policy="omit")  # LUNG goblet: no human block
lo2 = min(tc["sup_trained_auc_rat"].min(), tc["sup_trained_auc_human"].min()) - 0.02
hi2 = max(tc["sup_trained_auc_rat"].max(), tc["sup_trained_auc_human"].max()) + 0.02
ax.plot([lo2,hi2],[lo2,hi2], "k--", lw=1.0, alpha=0.4)
ax.axhline(0.65, color="#E74C3C", lw=1.2, ls=":", alpha=0.7, label="Preserved threshold (0.65)")
ax.set_xlabel("Rat trained AUC (PLS-1, species=2)", fontsize=10)
ax.set_ylabel("Human trained AUC (PLS-1, species=0)", fontsize=10)
ax.set_title(f"Global fidelity: exercise axis\nSpearman r={rho2:.3f} (p={pval2:.2e}, n={tc['sup_trained_auc_human'].notna().sum()})", fontsize=11)
ax.legend(fontsize=7, ncol=2, loc="upper left")
ax.text(0.98, 0.04, "●  hotspot (rat gate class)\n○  other", transform=ax.transAxes,
        ha="right", fontsize=8, color="#444")

plt.show()
print(f"sex validity: rat_med={med_r:.3f} human_med={med_h:.3f} r={rho:.3f}")
print(f"exercise fidelity Spearman: r={rho2:.3f}")


sex validity: rat_med=0.756 human_med=0.745 r=0.932
exercise fidelity Spearman: r=0.732


## Hotspot Survival: Descriptive Result

The next cells classify every Stage 11 hotspot in human space and draw the paired AUCs; the cell after the plots
prints the per-tissue counts. Over all blocks, the status counts and the rat-to-human Spearman of trained AUC and of
dose ρ are computed in the summary below. These classifications are descriptive and do not establish biological
conservation: no human measurement enters the comparison, and the human-space test re-reads the rat test's cells.

In [6]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

from nb_config import GATE_HUMAN_DIR
GCIH = str(GATE_HUMAN_DIR)
tc   = pd.read_csv(f"{GCIH}/transfer_comparison.tsv", sep="\t")
hot  = tc[tc["is_hotspot"]==True].copy()
hot  = hot.sort_values("sup_trained_auc_rat", ascending=False).reset_index(drop=True)

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "ovary":"#EC407A","spleen":"#B7950B",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}
STATUS_COLOR = {"PRESERVED":"#27AE60","LOST":"#E74C3C","WEAKENED":"#F39C12","no_human_block":"#BDC3C7"}

fig, axes = plt.subplots(1, 2, figsize=(15, 9), constrained_layout=True)

# Left: paired AUC bars (rat vs human) per hotspot
ax = axes[0]
y  = np.arange(len(hot))
w  = 0.35
rat_auc  = hot["sup_trained_auc_rat"].values
hum_auc  = hot["sup_trained_auc_human"].values
statuses = hot["status"].values

bars_r = ax.barh(y+w/2, rat_auc,  w, color="#2980B9", alpha=0.8, label="Rat (species=2)")
bars_h = ax.barh(y-w/2, hum_auc,  w,
                 color=[STATUS_COLOR.get(s,"#95A5A6") for s in statuses],
                 alpha=0.85, label="Human (species=0, color=status)")

ax.axvline(0.65, color="gray", lw=1.2, ls=":", alpha=0.7, label="Preserved threshold (0.65)")
ax.axvline(0.5,  color="gray", lw=0.8, ls="--", alpha=0.4)
ax.set_yticks(y)
labels = [f"{r['tissue']} / {r['cell_type']}" for _,r in hot.iterrows()]
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Supervised trained AUC (PLS-1)", fontsize=10)
ax.set_title(f"Exercise hotspot survival\n{len(hot)} hotspots (rat gate class); sorted by rat AUC", fontsize=11)
from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor="#27AE60", label=f"PRESERVED ({(statuses == 'PRESERVED').sum()}/{len(hot)})"),
    Patch(facecolor="#F39C12", label=f"WEAKENED ({(statuses == 'WEAKENED').sum()}/{len(hot)})"),
    Patch(facecolor="#E74C3C", label=f"LOST ({(statuses == 'LOST').sum()}/{len(hot)})"),
    Patch(facecolor="#2980B9", label="Rat AUC"),
]
ax.legend(handles=legend_els, fontsize=9, loc="lower right")

# Right: delta AUC waterfall
ax = axes[1]
delta_auc = hot["delta_auc"].values
colors    = [STATUS_COLOR.get(s,"#95A5A6") for s in statuses]
ax.barh(y, delta_auc, color=colors, alpha=0.85, edgecolor="k", lw=0.5)
ax.axvline(0, color="k", lw=0.8, alpha=0.6)
# status is set by the human-space AUC (>= 0.65) and p, not by the delta
ax.set_yticks(y)
ax.tick_params(axis="y", labelleft=False)
ax.set_xlabel("ΔAUC (human − rat)", fontsize=10)
ax.set_title("AUC change across species transfer\n(positive = amplified in human space)", fontsize=11)
from matplotlib.patches import Patch as P
ax.legend(handles=[
    P(facecolor="#27AE60", label="PRESERVED"),
    P(facecolor="#F39C12", label="WEAKENED"),
    P(facecolor="#E74C3C", label="LOST"),
], fontsize=9, loc="lower right")

plt.show()
print(hot["status"].value_counts().to_string())
print(f"delta_auc range: {delta_auc.min():.3f} to {delta_auc.max():.3f}")


status
PRESERVED    10
WEAKENED      1
delta_auc range: -0.120 to 0.095


In [7]:
import pandas as pd
from nb_config import GATE_HUMAN_DIR
GCIH = str(GATE_HUMAN_DIR)
tc   = pd.read_csv(f"{GCIH}/transfer_comparison.tsv", sep="\t")
hot  = tc[tc["is_hotspot"]==True]
summary = hot.groupby("tissue").agg(
    n_hotspot   = ("status","size"),
    n_preserved = ("status", lambda x: (x=="PRESERVED").sum()),
    n_lost      = ("status", lambda x: (x=="LOST").sum()),
    rat_auc_max = ("sup_trained_auc_rat","max"),
    human_auc_max=("sup_trained_auc_human","max"),
).reset_index()
print("=== per-tissue hotspot preservation ===")
print(summary.to_string(index=False))
print(f"\nTOTAL hotspots: {len(hot)}, PRESERVED: {(hot['status']=='PRESERVED').sum()}, "
      f"LOST: {(hot['status']=='LOST').sum()}")
print(f"WEAKENED: {(hot['status']=='WEAKENED').sum()}")
print("Human-space Augur was not run for v2 (nothing downstream reads it); see docs/RETRAIN_V2_RESULTS.md.")


=== per-tissue hotspot preservation ===
tissue  n_hotspot  n_preserved  n_lost  rat_auc_max  human_auc_max
 blood          4            4       0        0.826          0.855
 heart          1            0       0        0.812          0.696
  lung          2            2       0        0.819          0.896
 skmgn          1            1       0        0.838          0.718
 skmvl          3            3       0        0.884          0.846

TOTAL hotspots: 11, PRESERVED: 10, LOST: 0
WEAKENED: 1
Human-space Augur was not run for v2 (nothing downstream reads it); see docs/RETRAIN_V2_RESULTS.md.


In [8]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import spearmanr

from nb_config import GATE_HUMAN_DIR
GCIH = str(GATE_HUMAN_DIR)
tc   = pd.read_csv(f"{GCIH}/transfer_comparison.tsv", sep="\t")
tc["is_hotspot_bool"] = tc["is_hotspot"]==True

PALETTE = {
    "bat":"#D35400","blood":"#C0392B","cortex":"#8E44AD","heart":"#E74C3C",
    "hippoc":"#9B59B6","hypoth":"#6C3483","kidney":"#2980B9","liver":"#1ABC9C",
    "lung":"#27AE60","skmgn":"#F39C12","skmvl":"#E67E22","smlint":"#16A085",
    "ovary":"#EC407A","spleen":"#B7950B",
    "testes":"#5D6D7E","watsc":"#95A5A6",
}

fig, ax = plt.subplots(1, 1, figsize=(7.5, 5.5), constrained_layout=True)

# Rat vs human dose rho scatter (all blocks)
for t in sorted(tc["tissue"].unique()):
    sub = tc[tc["tissue"]==t]
    m   = sub["is_hotspot_bool"]
    ax.scatter(sub.loc[~m,"sup_dose_rho_rat"], sub.loc[~m,"sup_dose_rho_human"],
               c=PALETTE.get(t,"gray"), s=20, alpha=0.45, label=t.upper())
    ax.scatter(sub.loc[m,"sup_dose_rho_rat"],  sub.loc[m,"sup_dose_rho_human"],
               c=PALETTE.get(t,"gray"), s=65, edgecolors="k", lw=1.1, zorder=5)

rho, pval = spearmanr(tc["sup_dose_rho_rat"], tc["sup_dose_rho_human"], nan_policy="omit")  # LUNG goblet: no human block
lo = min(tc["sup_dose_rho_rat"].min(), tc["sup_dose_rho_human"].min()) - 0.05
hi = max(tc["sup_dose_rho_rat"].max(), tc["sup_dose_rho_human"].max()) + 0.05
ax.plot([lo,hi],[lo,hi], "k--", lw=1.0, alpha=0.3)
ax.axhline(0, color="gray", lw=0.8, ls="--", alpha=0.4)
ax.axvline(0, color="gray", lw=0.8, ls="--", alpha=0.4)
ax.set_xlabel("Rat dose ρ (ordinal, species=2)", fontsize=10)
ax.set_ylabel("Human dose ρ (ordinal, species=0)", fontsize=10)
ax.set_title(f"Ordinal dose-response transfer\nSpearman r={rho:.3f} (p={pval:.2e}, n={tc['sup_dose_rho_human'].notna().sum()})", fontsize=11)
ax.legend(fontsize=7, ncol=2, loc="upper left")
ax.text(0.98, 0.04, "●  hotspot\n○  other", transform=ax.transAxes, ha="right", fontsize=8, color="#444")

plt.show()
print(f"dose rho fidelity: r={rho:.3f}")
print("Rat-vs-human Augur panel omitted: human-space Augur was not run for v2.")

dose rho fidelity: r=0.794
Rat-vs-human Augur panel omitted: human-space Augur was not run for v2.


## Stage Summary

Stage 12 compares matched rat- and human-space blocks on the v2 embeddings of the pseudo-cells. The rat-to-human
embedding cosine mostly reflects the distance between the species' regions of the embedding, not whether a cell keeps
its identity; identity was tested against Tabula Sapiens instead, on the default before refs5
(`translation/ATLAS_PROJECTION_16_RESULTS.md`); that projection has not been repeated on the refs5 cells. The next cell computes the headline numbers for this config.

In [9]:
from scipy.stats import spearmanr
from IPython.display import Markdown, display
tc = pd.read_csv(GATE_HUMAN_DIR / "transfer_comparison.tsv", sep="\t")
hot = tc[tc["is_hotspot"] == True]
st = tc["status"].fillna("no_human_block").value_counts()
hs = hot["status"].value_counts()
both = tc.dropna(subset=["sup_trained_auc_human"])
sx = tc[["sup_sex_auc_rat", "sup_sex_auc_human"]].dropna()
ext = hot.sort_values("delta_auc")
# the corrected rule and the dose trend (columns written by compare_transfer.py since refs5)
corrected = ""
if "transfer_corrected" in tc.columns:
    names = lambda g: ", ".join(f"{t.upper()} {c}" for t, c in zip(g.tissue, g.cell_type)) or "none"
    corrected = (f" By the corrected rule (also called in human space under Benjamini-Hochberg) "
                 f"{int((hot['transfer_corrected'] == 'PASS').sum())} of {len(hot)} transfer; not: "
                 f"{names(hot[hot['transfer_corrected'] != 'PASS'])}. Dose trend lost in human space: "
                 f"{names(hot[hot['dose_trend_lost'] == True])}.")
    if "gate_class_rat" in tc.columns and (tc["gate_class_rat"] == "borderline").any():
        b = tc[tc["gate_class_rat"] == "borderline"]
        corrected += (" Borderline in rat space: " + "; ".join(
            f"{t.upper()} {c}: {s}, corrected rule {k}" for t, c, s, k in
            zip(b.tissue, b.cell_type, b.status, b.transfer_corrected)) + ".")
display(Markdown(f"""
**Results for this config:** {len(tc)} matched blocks.

- **Hotspots.** {hs.get('PRESERVED', 0)} of {len(hot)} preserved in human space by the registered rule
  ({hs.get('WEAKENED', 0)} weakened, {hs.get('LOST', 0)} lost).{corrected} Largest drop: {ext.iloc[0]['tissue']} {ext.iloc[0]['cell_type']}, AUC
  {ext.iloc[0]['sup_trained_auc_rat']:.3f} → {ext.iloc[0]['sup_trained_auc_human']:.3f}; largest gain:
  {ext.iloc[-1]['tissue']} {ext.iloc[-1]['cell_type']}, {ext.iloc[-1]['sup_trained_auc_rat']:.3f} →
  {ext.iloc[-1]['sup_trained_auc_human']:.3f} (delta-AUC range {hot['delta_auc'].min():+.3f} to {hot['delta_auc'].max():+.3f}).
- **All blocks.** {", ".join(f"{v} {k.lower().replace('_', ' ')}" for k, v in st.items())}.
- **Sex check.** Median sex AUC rat {sx.iloc[:, 0].median():.3f} → human {sx.iloc[:, 1].median():.3f} (floor 0.70, set in the script);
  block-level Spearman {spearmanr(sx.iloc[:, 0], sx.iloc[:, 1]).correlation:.3f} over {len(sx)} blocks.
- **Ordering.** Human-space trained AUCs follow rat-space AUCs at Spearman
  {spearmanr(both['sup_trained_auc_rat'], both['sup_trained_auc_human']).correlation:.3f} over the {len(both)} blocks
  with a human block, and dose ρ at {spearmanr(both['sup_dose_rho_rat'], both['sup_dose_rho_human'], nan_policy='omit').correlation:.3f}.
"""))


**Results for this config:** 194 matched blocks.

- **Hotspots.** 10 of 11 preserved in human space by the registered rule
  (1 weakened, 0 lost). By the corrected rule (also called in human space under Benjamini-Hochberg) 9 of 11 transfer; not: SKMGN Schwann cells, HEART Vascular smooth muscle cells. Dose trend lost in human space: LUNG Vascular smooth muscle cells. Borderline in rat space: SKMGN Vascular smooth muscle cells: WEAKENED, corrected rule FAIL. Largest drop: skmgn Schwann cells, AUC
  0.838 → 0.718; largest gain:
  lung Endothelial cells, 0.801 →
  0.896 (delta-AUC range -0.120 to +0.095).
- **All blocks.** 119 lost, 45 preserved, 29 weakened, 1 no human block.
- **Sex check.** Median sex AUC rat 0.756 → human 0.745 (floor 0.70, set in the script);
  block-level Spearman 0.932 over 170 blocks.
- **Ordering.** Human-space trained AUCs follow rat-space AUCs at Spearman
  0.732 over the 193 blocks
  with a human block, and dose ρ at 0.794.


## Reproducibility Notes

The default tree's transfer (re-expression as human at the config's `pseudocell_target_sum`, embedding with species 0,
the human-space probe by the gate method and `compare_transfer.py`) ran in the refs5 chain job 12377824 (commit
`44332e6`, no modified tracked file). The transfer of the default before refs5 was job 12122211; the refs4 (14-tissue)
transfer was job 12047419; the July-reference v2 run is jobs 12014308/12014934/12014935.

```bash
python pipeline/run_stage12.py --jobs 48   # v2 with all 16 tissues is the default config
#   steps 1-2: transfer + embed per tissue
#   step 3: human-space probe by the gate method (10 splits; 10,000 permutations, then sequential batches; the rat
#           gate's family of blocks) -> <genecompass_input_dir>_human/gate_rcv10/
#   step 4: compare_transfer.py against the rat gate (deconvolution.hotspots_table's folder) -> same folder
python pipeline/run_stage12.py --from 3 --jobs 48   # steps 3-4 only
```

Interpretation constraints:

1. Transfer is one-way from rat expression into human vocabulary; no human exercise dataset is used.
2. Hotspot preservation values are descriptive, even when the sex check passes. The human-space test reuses the rat
   test's cells, labels, folds and seeds, so it is not a replication.

In [10]:
import pandas as pd, hashlib, pathlib, numpy as np

from nb_config import GC_HUMAN_DIR as GCIH
GATE = GCIH / "gate_rcv10"
for fn in ["transfer_comparison.tsv","subspace_probe.tsv"]:
    p = GATE / fn
    md5 = hashlib.md5(p.read_bytes()).hexdigest()
    nrows = len(open(p).readlines()) - 1
    print(f"  {fn}: {nrows} rows | MD5={md5}")

tc = pd.read_csv(GATE / "transfer_comparison.tsv", sep="\t")
hot = tc[tc["is_hotspot"]==True]
print(f"\n  total blocks: {len(tc)}")
print(f"  hotspot blocks: {len(hot)}")
print(f"  PRESERVED: {(hot['status']=='PRESERVED').sum()}")
print(f"  WEAKENED: {(hot['status']=='WEAKENED').sum()}")
print(f"  LOST: {(hot['status']=='LOST').sum()}")
print(f"  sex validity (human med): {tc['sup_sex_auc_human'].median():.3f}")
print(f"  sex r(rat,human): {tc['sup_sex_auc_rat'].corr(tc['sup_sex_auc_human'], method='spearman'):.3f}")

from nb_config import tissues
total = 0
for t in tissues:
    emb = GCIH / t / "embeddings" / "cell_embeddings.npy"
    if emb.exists():
        arr = np.load(str(emb), mmap_mode="r")
        total += arr.shape[0]
print(f"\n  total human-space embeddings: {total} cells × 768 dims")


  transfer_comparison.tsv: 194 rows | MD5=f3d2abaeb1179c5844af40fdf4672746
  subspace_probe.tsv: 194 rows | MD5=c7a7cf826336004caf819f2fc79a4d3c

  total blocks: 194
  hotspot blocks: 11
  PRESERVED: 10
  WEAKENED: 1
  LOST: 0
  sex validity (human med): 0.745
  sex r(rat,human): 0.932

  total human-space embeddings: 9041 cells × 768 dims
